# 01. Data and threat models

The datasets the study uses: Table II (`reports/tables/datasets.csv`). Then one dataset as the library loads it,
with its feature schema (which features the attacker controls), and the three threat models applied live to a few
test frauds.

**Inputs:** `data/processed/<dataset>/meta.json` and `arrays.npz` (written by `fraudadv.data.prepare`), and the
shared fraud sample `runs/<dataset>/attacks/sample.npz` of each dataset.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))   # the library (not needed after pip install -e .)
from fraudadv import notebook as nb
ROOT = nb.setup()          # works from the folder root, sets the study's environment

## Table II: datasets

`tables.datasets` counts population rows and frauds over the three out-of-time splits, the features and
controllable features of each dataset (from `meta.json`), and the size of the fraud sample every model is attacked
on (from `sample.npz`), into `reports/tables/datasets.csv`.

In [ ]:
from fraudadv.report import tables

tables.datasets()
nb.show_csv("reports/tables/datasets.csv")

## One dataset as the models and attacks see it

`fraudadv.data.load.load` returns the arrays of each split (standardized features `X_`, labels `y_`, sampling
weights `w_`, dollar amounts `amt_`, times `t_`, entity ids `ent_`) and the metadata. Every model, attack and
analysis reads its data this way. Here: Sparkov.

In [ ]:
import numpy as np
import pandas as pd
from fraudadv.data.load import load

DATASET = "sparkov"
a, meta = load(DATASET)
display(pd.DataFrame([(k, v.shape, v.dtype) for k, v in a.items()], columns=["array", "shape", "dtype"]))
pd.DataFrame(meta["sizes"]).T

The feature schema of the dataset. `controllable` are the features in `meta["mutable"]`,
the only ones the constrained and consistent attackers may change. The bounds are the training range, shown here
in original units through the scaler in `meta.json`. `meta["recompute"]` names the derived-feature rule
(Sparkov: `log_ratio` follows `log_amt`).

In [ ]:
mu, sd = np.array(meta["mu"]), np.array(meta["sd"])
schema = pd.DataFrame({
    "feature": meta["features"],
    "controllable": [f in meta["mutable"] for f in meta["features"]],
    "categorical": [f in meta["categorical"] for f in meta["features"]],
    "integer": [f in meta["integer"] for f in meta["features"]],
    "min (raw)": np.array(meta["lower"]) * sd + mu,
    "max (raw)": np.array(meta["upper"]) * sd + mu,
})
print("controllable features:", meta["mutable"])
print("derived-feature rule:", meta["recompute"])
schema.round(3)

## Run it yourself: the three threat models on a few frauds

`Constraints(meta, X_train, mode)` builds each threat model; its `project(X, X0, eps)` maps candidate rows back
onto it after every attack step. Here five test frauds of Sparkov get a random perturbation of every feature
within $\epsilon = 0.25$ (standardized units), and each threat model projects it:

* **unconstrained**: every feature may move inside the data domain (discrete features snap to valid values,
  derived features are recomputed);
* **constrained**: only the controllable features move;
* **consistent**: as constrained, then repaired so the row is one a real transaction could produce.

`fraudadv.attacks.realizable.violations_all` flags rows no real transaction could produce (the validator behind
Table S-XIII). This is an illustration on five random frauds, not a result of the study; nothing is written to
disk.

In [ ]:
from fraudadv.attacks.constraints import Constraints
from fraudadv.attacks.realizable import violations_all

EPS = 0.25
rng = np.random.default_rng(0)
frauds = np.flatnonzero(a["y_test"] == 1)
X0 = a["X_test"][rng.choice(frauds, 5, replace=False)]
X_try = X0 + rng.uniform(-EPS, EPS, X0.shape).astype(np.float32)      # a random step inside the budget

features = np.array(meta["features"])
rows, projected = [], {}
for mode in ("unconstrained", "constrained", "consistent"):
    cons = Constraints(meta, a["X_train"], mode)
    X = cons.project(X_try, X0, EPS)
    projected[mode] = X
    bad = violations_all(DATASET, X)
    for i in range(len(X)):
        moved = features[np.abs(X[i] - X0[i]) > 0]
        rows.append(dict(mode=mode, fraud=i, linf=round(float(cons.linf(X[i:i + 1], X0[i:i + 1])[0]), 3),
                         n_moved=len(moved), impossible=bool(bad[i]), moved=", ".join(moved)))
print("original rows flagged impossible:", violations_all(DATASET, X0).tolist())
pd.DataFrame(rows)

The constrained attacker moves the merchant's coordinates and the distance `dist` independently, although `dist`
is derived from the merchant's and the cardholder's locations, so its rows can be impossible (the unconstrained
attacker's too); the consistent attacker's repair makes them realizable: a merchant the simulator could produce
and the distance that goes with it. One fraud in original units, controllable features only:

In [ ]:
cols = [meta["features"].index(f) for f in meta["mutable"]]
raw = lambda X: X[0, cols] * sd[cols] + mu[cols]
pd.DataFrame({"original": raw(X0), **{m: raw(X) for m, X in projected.items()}}, index=meta["mutable"]).round(4)

In [ ]:
assert not violations_all(DATASET, projected["consistent"]).any(), "a consistent row is not realizable"
print("every consistent row passes violations_all")

## Rerun the runs behind these results from scratch

The data step of `scripts/reproduce.sh` (`python -m fraudadv.data.prepare <dataset> [--context]`, skipped when
`arrays.npz` exists): raw files in `data/raw/<dataset>/` (sources in `DATA.md`) become `data/processed/<dataset>/`.
It takes minutes to an hour per dataset and writes into `data/processed/`; the fraud sample of Table II is drawn
by the attack grid (notebook of the attack grid, `fraudadv.attacks.run_attacks`). Not executed here.

In [ ]:
RERUN = False
DATASET = "ulb"
if RERUN:
    from fraudadv.data import prepare
    from fraudadv.paths import PROC

    CONTEXT = DATASET in ("sparkov", "tabformer", "ieee_cis", "paysim")      # datasets with card/account histories
    if not (PROC / DATASET / "arrays.npz").exists():
        prepare.finalize(prepare.BUILDERS[DATASET](), context=CONTEXT)

In [ ]:
ok = nb.check(["reports/tables/datasets.csv"])